# 🍷 Wine Quality Prediction

Predicting wine quality from chemical characteristics using three classifier models:
- **Random Forest Classifier**
- **Stochastic Gradient Descent (SGD)**
- **Support Vector Classifier (SVC)**

---

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('All libraries loaded successfully!')

## 2. Load & Explore Dataset

In [ ]:
df = pd.read_csv('../data/WineQT.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

## 3. Data Visualization

In [ ]:
# Quality distribution
fig, ax = plt.subplots(figsize=(8, 5))
quality_counts = df['quality'].value_counts().sort_index()
bars = ax.bar(quality_counts.index, quality_counts.values,
              color=sns.color_palette('muted', len(quality_counts)))
ax.bar_label(bars, padding=3)
ax.set_xlabel('Quality Score', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Wine Quality Score Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/quality_distribution.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 9))
corr = df.drop(columns=['Id']).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Key chemical features vs Quality
key_features = ['density', 'volatile acidity', 'alcohol', 'citric acid']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    sns.boxplot(x='quality', y=feat, data=df, ax=axes[i], palette='muted')
    axes[i].set_title(f'{feat.title()} vs Quality', fontweight='bold')
    axes[i].set_xlabel('Quality Score')
    axes[i].set_ylabel(feat.title())

plt.suptitle('Key Chemical Features vs Wine Quality', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../visualizations/features_vs_quality.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature distributions
features = [c for c in df.columns if c not in ['quality', 'Id']]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].hist(df[feat], bins=30, color=sns.color_palette('muted')[i % 6], edgecolor='white')
    axes[i].set_title(feat.title(), fontweight='bold', fontsize=10)
    axes[i].set_xlabel('')

# Hide any unused subplots
for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribution of All Chemical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/feature_distributions.png', dpi=150)
plt.show()

## 4. Data Preprocessing

In [ ]:
# Drop the Id column (not a feature)
df = df.drop(columns=['Id'])

# Features and target
X = df.drop(columns=['quality'])
y = df['quality']

print(f'Features: {X.shape[1]} columns')
print(f'Target classes: {sorted(y.unique())}')

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTraining samples : {X_train.shape[0]}')
print(f'Testing samples  : {X_test.shape[0]}')

# Feature scaling (important for SGD and SVC)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('\nFeature scaling applied (StandardScaler).')

## 5. Model Training & Evaluation

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Train, predict, and display evaluation metrics for a classifier."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)

    print(f'\n{"=" * 55}')
    print(f'  {name}')
    print(f'  Accuracy: {acc:.4f} ({acc*100:.2f}%)')
    print(f'{"=" * 55}')
    print(classification_report(y_te, y_pred, zero_division=0))

    # Confusion matrix
    fig, ax = plt.subplots(figsize=(7, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_te, y_pred, ax=ax,
        colorbar=False, cmap='Blues'
    )
    ax.set_title(f'Confusion Matrix — {name}', fontweight='bold')
    plt.tight_layout()
    fname = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    plt.savefig(f'../visualizations/cm_{fname}.png', dpi=150)
    plt.show()

    return acc, model

### 5.1 Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=None,
                             random_state=42, n_jobs=-1)
rf_acc, rf_model = evaluate_model(
    'Random Forest Classifier', rf,
    X_train, X_test, y_train, y_test
)

In [ ]:
# Feature importances (Random Forest)
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors = sns.color_palette('muted', len(importances))
ax.barh(importances.index, importances.values, color=colors)
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Feature Importances — Random Forest', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/feature_importances.png', dpi=150)
plt.show()

### 5.2 Stochastic Gradient Descent (SGD)

In [ ]:
sgd = SGDClassifier(max_iter=1000, tol=1e-3, random_state=42)
sgd_acc, sgd_model = evaluate_model(
    'SGD Classifier', sgd,
    X_train_scaled, X_test_scaled, y_train, y_test
)

### 5.3 Support Vector Classifier (SVC)

In [ ]:
svc = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svc_acc, svc_model = evaluate_model(
    'Support Vector Classifier (SVC)', svc,
    X_train_scaled, X_test_scaled, y_train, y_test
)

## 6. Model Comparison

In [ ]:
results = {
    'Random Forest': rf_acc,
    'SGD Classifier': sgd_acc,
    'SVC (RBF kernel)': svc_acc,
}

results_df = pd.DataFrame.from_dict(results, orient='index', columns=['Accuracy'])
results_df['Accuracy (%)'] = (results_df['Accuracy'] * 100).round(2)
results_df = results_df.sort_values('Accuracy', ascending=False)
print(results_df.to_string())

# Bar chart comparison
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71' if v == results_df['Accuracy'].max() else '#3498db'
          for v in results_df['Accuracy'].values]
bars = ax.bar(results_df.index, results_df['Accuracy'] * 100, color=colors, width=0.5)
ax.bar_label(bars, fmt='%.2f%%', padding=4, fontsize=11)
ax.set_ylim(0, 105)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
ax.axhline(y=results_df['Accuracy'].max() * 100, color='gray',
           linestyle='--', linewidth=1, alpha=0.6)
plt.tight_layout()
plt.savefig('../visualizations/model_comparison.png', dpi=150)
plt.show()

## 7. Conclusion

| Model | Accuracy |
|-------|----------|
| Random Forest | Best performer — captures non-linear relationships among chemical features |
| SVC (RBF) | Strong second — effective in high-dimensional feature spaces |
| SGD Classifier | Fastest to train — useful baseline for large-scale data |

**Key chemical predictors** (from Random Forest importances): `alcohol`, `volatile acidity`, `sulphates`, and `density` are the strongest signals of wine quality.

### Next Steps
- Try hyperparameter tuning with `GridSearchCV` or `RandomizedSearchCV`
- Experiment with class balancing (SMOTE) to handle the skewed quality distribution
- Explore XGBoost or LightGBM for potentially higher accuracy